# **TFM: Detecció d'esdeveniments importants en partits de futbol a partir de les seves narracions**

**Autor:** Martí Mullor Rordíguez

**Institució:** Universitat Oberta de Catalunya  
**Tutor:** Josep Mª Carmona Leyva

**Data:** Juny 2026

---

### **Nom de l'script: 01_Preproces**

En aquest script es dur a terme el preprocés de les dades.

Està compost per:

**0. Importacions**

**1. Carregar la configuració inicial**

**2. Funcions d'alineació i parsing**

**3. Bucle de processament**

**4. Dvisió del dataset**






---


## **0. Importacions**

In [ ]:
import os
import sys
import json
import glob
import shutil
import subprocess
import pandas as pd
import numpy as np
import librosa
import torch
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from google.colab import drive

## **1. Carregar la configuració inicial**

### **1.1. Muntar Google Drive**



In [ ]:
if not os.path.exists('/content/drive'):
    print("Muntant Google Drive")
    drive.mount('/content/drive')

### **1.2. Carregar configuració inicial**

In [ ]:
CONFIG_PATH = "/content/drive/MyDrive/TFM/TFM-Deteccio-Esdeveniments-Futbol/config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

paths = config["paths"]
seed = config["global_settings"]["random_seed"]
settings = config["fase_01_preproces"]

LOCAL_RAW_DIR = paths["local_raw"]
LOCAL_TEMP_ECHOES = paths["local_temp_echoes"]
AUDIO_MODEL_NAME = settings.get("audio_embedding_model", "facebook/wav2vec2-base")
TARGET_SR = settings.get("target_sample_rate", 16000)

os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

## **2. Descàrrega de les etiquetes, transcripcions i vídeo**

In [ ]:
!pip install SoccerNet -q
from SoccerNet.Downloader import SoccerNetDownloader

# Inicialització del descarregador oficial
downloader = SoccerNetDownloader(LocalDirectory=paths["raw_data"])
downloader.password = settings["soccernet_password"]

In [ ]:
downloader_local = SoccerNetDownloader(LocalDirectory=LOCAL_RAW_DIR)
downloader_local.password = settings["soccernet_password"]

In [ ]:
# 2.A. Descàrrega de les Etiquetes de SoccerNet-v2
labels_check = glob.glob(os.path.join(LOCAL_RAW_DIR, "**/Labels-v2.json"), recursive=True)
if len(labels_check) == 0:
    print("No s'han trobat etiquetes locals. Descarregant Labels-v2.json...")
    # Descàrrega dels splits principals
    downloader.downloadGames(files=["Labels-v2.json"], split=["train", "valid", "test"])
    print("Etiquetes desades a la carpeta RAW.")


In [ ]:
# 2.B. Descàrrega de les Transcripcions de Text (SoccerNet-Echoes) - FUSIÓ NETEJA

WHISPER_VERSION = settings["whisper_version"]
ECHOES_GIT_URL = settings["echoes_git_url"]

json_files = glob.glob(os.path.join(LOCAL_RAW_DIR, "**/*.json"), recursive=True)
commentary_jsons = [j for j in json_files if "Labels" not in os.path.basename(j)]

if len(commentary_jsons) == 0:
    print(f" Descarregant SoccerNet-Echoes ({WHISPER_VERSION}) des del repositori...")

    if os.path.exists(LOCAL_TEMP_ECHOES):
        shutil.rmtree(LOCAL_TEMP_ECHOES)

    # Clonació del repostiroi de manera temporal
    !git clone {ECHOES_GIT_URL} {LOCAL_TEMP_ECHOES} -q

    src_dataset_specific = os.path.join(LOCAL_TEMP_ECHOES, "Dataset", WHISPER_VERSION)

    if os.path.exists(src_dataset_specific):
        print(f"Fusionant les lligues de {WHISPER_VERSION} amb l'estructura mestre del dataset...")
        !cp -r {src_dataset_specific}/* {LOCAL_RAW_DIR}/
        print("Estructura unificada correctament en local: lliga/temporada/partit")


        shutil.rmtree(LOCAL_TEMP_ECHOES)
    else:
        print(f"ERROR: No s'ha trobat la carpeta {WHISPER_VERSION} al repositori clonat.")
else:
    print(f"S'han detectat {len(commentary_jsons)} fitxers de comentaris de text ja alineats.")


In [ ]:
# 2.C. Descàrrega dels Vídeos/Àudios reals (Opcional i sota demanda)
if settings.get("download_videos", False):
    res = settings["video_resolution"]
    print(f"S'ha activat la descàrrega de vídeos ({res}) al disc LOCAL.")
    try:
        # Descarregador local en lloc del de Drive
        downloader_local.downloadGames(files=[f"1_{res}.mkv", f"2_{res}.mkv"], split=["valid"])
        print("Descàrrega/Verificació de fitxers de vídeo finalitzada en local.")
    except Exception as e:
        print(f"Nota de descàrrega de vídeo: {e}. Comprova el password si ha fallat.")

## **3. Funcions d'àudio i models**

In [ ]:
def extract_wav_from_video(video_path, output_wav_path):
    """Executa FFmpeg per extreure l'àudio en format WAV si es disposa del vídeo."""
    if os.path.exists(output_wav_path) or not os.path.exists(video_path):
        return
    print(f"Extreient àudio des de: {os.path.basename(video_path)}")
    command = f"ffmpeg -i '{video_path}' -vn -acodec pcm_s16le -ar 16000 -ac 1 '{output_wav_path}' -y -loglevel quiet"
    subprocess.run(command, shell=True)

In [ ]:
print(f"Carregant el model de característiques acústiques: {AUDIO_MODEL_NAME}...")

audio_processor = Wav2Vec2Processor.from_pretrained(AUDIO_MODEL_NAME)
audio_model = Wav2Vec2Model.from_pretrained(AUDIO_MODEL_NAME).to("cuda")

In [ ]:
def get_audio_embedding(wav_path, start_sec, end_sec):
    try:
        duration = end_sec - start_sec
        speech, _ = librosa.load(wav_path, sr=TARGET_SR, offset=start_sec, duration=duration)
        inputs = audio_processor(speech, sampling_rate=TARGET_SR, return_tensors="pt")

        # Enviament de les dades a l'accelerador gràfic actiu
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            outputs = audio_model(**inputs)

        return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    except Exception as e:
        return None

## **4. Alineació temporal**

In [ ]:
def parse_soccernet_labels(labels_json_path):
    events = []
    if not os.path.exists(labels_json_path): return events
    with open(labels_json_path, "r", encoding="utf-8") as f: data = json.load(f)
    for annotation in data.get("annotations", []):
        game_time = annotation.get("gameTime", "")
        half = int(game_time.split(" - ")[0]) if " - " in game_time else 1
        position_sec = int(annotation.get("position", 0)) / 1000.0
        events.append({"half": half, "seconds": position_sec, "label": annotation.get("label", "Unknown")})
    return events

def align_half_transcription(asr_json_path, events, current_half, audio_path, bg_label):
    rows = []
    if not os.path.exists(asr_json_path): return rows
    with open(asr_json_path, "r", encoding="utf-8") as f: echoes_data = json.load(f)

    segments = echoes_data.get("segments", {})
    half_events = [ev for ev in events if ev["half"] == current_half]
    has_audio = os.path.exists(audio_path)

    for idx, seg_data in segments.items():
        if isinstance(seg_data, list):
            start, end, text = float(seg_data[0]), float(seg_data[1]), str(seg_data[2]).strip()
        elif isinstance(seg_data, dict):
            start, end, text = float(seg_data.get("start_time", 0.0)), float(seg_data.get("end_time", 0.0)), str(seg_data.get("text", "")).strip()
        else: continue
        if not text: continue

        final_label = bg_label
        for ev in half_events:
            if start <= ev["seconds"] <= end:
                final_label = ev["label"]
                break

        audio_emb = get_audio_embedding(audio_path, start, end) if has_audio else None

        rows.append({
            "text": text, "start_time": start, "end_time": end,
            "half": current_half, "label": final_label,
            "has_audio": has_audio,
            "audio_vector": audio_emb
        })
    return rows

## **5. Processament**

In [ ]:
all_segments_list = []

drive_raw_dir = config["paths"]["raw_data"]
labels_files = glob.glob(os.path.join(drive_raw_dir, "**/Labels-v2.json"), recursive=True)

print(f"\nComençant l'alineació dels {len(labels_files)} partits trobats al Drive...")

In [ ]:
total_vectors = 0
vectors_exits = 0
vectors_fallits = 0

for labels_path in tqdm(labels_files, desc="Processant partits"):
    game_drive_dir = os.path.dirname(labels_path)
    rel_path = os.path.relpath(game_drive_dir, drive_raw_dir)
    game_id = rel_path
    game_local_dir = os.path.join(LOCAL_RAW_DIR, rel_path)

    game_events = parse_soccernet_labels(labels_path)
    if not game_events: continue

    all_jsons = glob.glob(os.path.join(game_local_dir, "**/*.json"), recursive=True)
    asr_files = [j for j in all_jsons if "Labels" not in os.path.basename(j)]

    for asr_path in asr_files:
        filename = os.path.basename(asr_path)
        half_num = 1 if "1" in filename else (2 if "2" in filename else None)
        if half_num is None: continue

        video_source = os.path.join(game_local_dir, f"{half_num}_{settings['video_resolution']}.mkv")
        audio_target = os.path.join(game_local_dir, f"{half_num}{settings['audio_extension']}")

        if settings.get("extract_audio_ffmpeg", True) and os.path.exists(video_source):
            extract_wav_from_video(video_source, audio_target)

        half_rows = align_half_transcription(asr_path, game_events, half_num, audio_target, settings["background_label"])

        for row in half_rows:
            row["game_id"] = game_id
            all_segments_list.append(row)

            # Checkpoint en temps real
            if row["has_audio"]:
                total_vectors += 1
                if row["audio_vector"] is not None:
                    vectors_exits += 1
                else:
                    vectors_fallits += 1

                if total_vectors % 500 == 0:
                    print(f"\n--- CHECKPOINT: {total_vectors} vectors acústics ---")
                    print(f"Processats (GPU): {vectors_exits} | Fallits: {vectors_fallits}")

        if os.path.exists(audio_target):
            os.remove(audio_target)

df_master = pd.DataFrame(all_segments_list)
print(f"\nProcés completat! Mida del dataset resultant: {df_master.shape}")

# Verificació ràpida de l'estructura final
display(df_master[df_master["has_audio"] == True].head(3))
print(f"Vectors buits restants: {df_master['audio_vector'].isna().sum()}")

In [ ]:
print(df_master['audio_vector'].isna().sum())

## **6. Split grupal dinámic i exportació**

In [ ]:
if len(df_master) > 0:
    unique_games = df_master["game_id"].unique()
    np.random.seed(seed)
    np.random.shuffle(unique_games)

    n_games = len(unique_games)
    train_idx = int(n_games * settings["train_ratio"])
    val_idx = int(n_games * (settings["train_ratio"] + settings["val_ratio"]))

    train_games = unique_games[:train_idx]
    val_games = unique_games[train_idx:val_idx]
    test_games = unique_games[val_idx:]

    df_train = df_master[df_master["game_id"].isin(train_games)].copy()
    df_val = df_master[df_master["game_id"].isin(val_games)].copy()
    df_test = df_master[df_master["game_id"].isin(test_games)].copy()

    print("\n================ Validació de l'Estructura ================")
    print(f"  • Conjunt d'Entrenament (Train) : {len(df_train)} fragments ({len(train_games)} partits)")
    print(f"  • Conjunt de Validació (Val)     : {len(df_val)} fragments ({len(val_games)} partits)")
    print(f"  • Conjunt de Test (Test)         : {len(df_test)} fragments ({len(test_games)} partits)")
    print("===========================================================\n")

    os.makedirs(paths["processed_data"], exist_ok=True)

    # Exportació  a Google Drive
    df_train.to_pickle(os.path.join(paths["processed_data"], "train_dataset.pkl"))
    df_val.to_pickle(os.path.join(paths["processed_data"], "val_dataset.pkl"))
    df_test.to_pickle(os.path.join(paths["processed_data"], "test_dataset.pkl"))
    print(f"Processat completat des de zero. Fitxers guardats de forma permanent a Drive: {paths['processed_data']}\n")
else:
    print("ERROR CRÍTIC: No s'ha pogut generar cap segment d'informació. Revisa la descàrrega en local.")